> 在线讲义：[第 54 章 · 点云与配准：KD 树、法向量、Procrustes / Kabsch、ICP](https://wikieden.github.io/math-and-eval-stats/lessons/54-point-cloud.html) · 本书《数学与实验统计》

# 第 54 章 · 点云与配准：KD 树、法向量、Procrustes / Kabsch、ICP

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/54-point-cloud.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 KD 树 | cKDTree | `nn_compare()` | 和暴力一致 |
| 第二步 法向量 | 最小特征向量 | `normal_from_neighbors()` | 夹角 0.15° |
| 第三步 Kabsch | SVD + det | `kabsch(p, q, fix)` | det −1 → 1 |
| 第四步 残差 | 点到点 / 点到面 | `residuals(p, q, n)` | 0.1 / 0 |
| 第五步 ICP 一步 | 最近点 + Kabsch | `icp_one_step()` | 0.268 → 0.227 |

例子：KD 树找最近邻（和暴力比较次数）；法向量 = 局部协方差最小特征值对应的特征向量；Kabsch / Procrustes 的反射修正；
点到点与点到面的残差；ICP 的一次“找对应 → 求解”。
里程碑 ME-M33 的“Kabsch 求旋转、完整的点到点 ICP 循环”留给自己做；这里只演示公式和一步。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

**函数 `nn_compare`**：暴力最近邻要算 n 次距离；KD 树平均只访问 O(log n) 个节点。返回 (最大距离差, 暴力的距离计算次数)。

In [ ]:
def nn_compare(n: int = 20000, q: int = 200, seed: int = 0) -> tuple[float, float]:
    """暴力最近邻要算 n 次距离；KD 树平均只访问 O(log n) 个节点。返回 (最大距离差, 暴力的距离计算次数)。"""
    rng = np.random.default_rng(seed)
    pts, qs = rng.uniform(0, 10, (n, 3)), rng.uniform(0, 10, (q, 3))
    d_tree, _ = cKDTree(pts).query(qs)
    d_brute = np.sqrt(((qs[:, None, :] - pts[None]) ** 2).sum(-1)).min(1)
    return float(np.abs(d_tree - d_brute).max()), n

**函数 `normal_from_neighbors`**：平面 z = 0.2x + 0.1y 上 50 个点加小噪声：协方差最小特征值对应的特征向量就是法向量。返回 (估计, 真值)。

In [ ]:
def normal_from_neighbors(seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """平面 z = 0.2x + 0.1y 上 50 个点加小噪声：协方差最小特征值对应的特征向量就是法向量。返回 (估计, 真值)。"""
    rng = np.random.default_rng(seed)
    xy = rng.uniform(-1, 1, (50, 2))
    z = 0.2 * xy[:, 0] + 0.1 * xy[:, 1] + 0.005 * rng.standard_normal(50)
    p = np.c_[xy, z]
    w, v = np.linalg.eigh(np.cov((p - p.mean(0)).T))
    n = v[:, 0] * np.sign(v[2, 0])
    true = np.array([-0.2, -0.1, 1.0])
    return n, true / np.linalg.norm(true)

**函数 `kabsch`**：找 R 使 Σ‖R p_i − q_i‖² 最小（点已去中心）：H = Σ p qᵀ = UΣVᵀ，R = V diag(1, 1, det(VUᵀ)) Uᵀ。

In [ ]:
def kabsch(p: np.ndarray, q: np.ndarray, fix_reflection: bool = True) -> np.ndarray:
    """找 R 使 Σ‖R p_i − q_i‖² 最小（点已去中心）：H = Σ p qᵀ = UΣVᵀ，R = V diag(1, 1, det(VUᵀ)) Uᵀ。"""
    u, _, vt = np.linalg.svd(p.T @ q)
    d = np.sign(np.linalg.det(vt.T @ u.T)) if fix_reflection else 1.0
    return vt.T @ np.diag([1, 1, d]) @ u.T

**函数 `residuals`**：点到点 ‖p − q‖ 与点到面 |(p − q)·n|：沿表面滑动的误差点到面不罚。

In [ ]:
def residuals(p: np.ndarray, q: np.ndarray, n: np.ndarray) -> tuple[float, float]:
    """点到点 ‖p − q‖ 与点到面 |(p − q)·n|：沿表面滑动的误差点到面不罚。"""
    return float(np.linalg.norm(p - q)), float(abs((p - q) @ n))

**函数 `icp_one_step`**：目标是一个随机点云，源 = 目标转 15°、平移 0.1。最近邻找对应 → 去中心 → Kabsch → 更新。返回 (初始 RMS, 一步后 RMS, 一步后的残余转角)。

In [ ]:
def icp_one_step(seed: int = 0) -> tuple[float, float, float]:
    """目标是一个随机点云，源 = 目标转 15°、平移 0.1。最近邻找对应 → 去中心 → Kabsch → 更新。返回 (初始 RMS, 一步后 RMS, 一步后的残余转角)。"""
    rng = np.random.default_rng(seed)
    tgt = rng.uniform(-1, 1, (300, 3))
    th = np.radians(15)
    r = np.array([[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]])
    src = tgt @ r.T + 0.1
    _, idx = cKDTree(tgt).query(src)
    q = tgt[idx]
    ps, qs = src - src.mean(0), q - q.mean(0)
    rr = kabsch(ps, qs)
    new = (src - src.mean(0)) @ rr.T + q.mean(0)
    _, idx2 = cKDTree(tgt).query(new)
    rest = rr @ r
    ang = np.degrees(np.arccos(np.clip((np.trace(rest) - 1) / 2, -1, 1)))
    return float(np.sqrt(((src - tgt) ** 2).sum(1).mean())), float(np.sqrt(((new - tgt) ** 2).sum(1).mean())), float(ang)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 最近邻与 KD 树")
d, n = nn_compare()
print(f"  2 万个点、200 次查询：KD 树和暴力结果最大差 {d:.1e}；暴力每次要算 {n} 个距离，KD 树平均只看几十个节点")

In [ ]:
print("\n第二步 法向量 = 局部协方差最小特征值的特征向量")
nn_, true = normal_from_neighbors()
print(f"  平面 z = 0.2x + 0.1y：估计 {np.round(nn_, 4).tolist()}，真值 {np.round(true, 4).tolist()}，夹角 {np.degrees(np.arccos(nn_ @ true)):.2f}°")

In [ ]:
print("\n第三步 Kabsch：SVD 求最优旋转，注意反射")
p = np.array([[1.0, 0, 0], [0, 1, 0], [0, 0, 1], [-1, -1, -1]]) * 0.5
q = p * np.array([1, 1, -1])            # 镜像：最优“正交矩阵”是反射
r0, r1 = kabsch(p, q, False), kabsch(p, q, True)
print(f"  目标是源的镜像：不修正 det = {np.linalg.det(r0):.0f}（反射，不是旋转）；修正后 det = {np.linalg.det(r1):.0f}")

In [ ]:
print("\n第四步 点到点 vs 点到面")
for name, pp in (("沿表面错开 0.1", np.array([0.1, 0, 0])), ("离开表面 0.1", np.array([0, 0, 0.1]))):
    a, b = residuals(pp, np.zeros(3), np.array([0, 0, 1.0]))
    print(f"  {name}：点到点 {a:.3f}，点到面 {b:.3f}")

In [ ]:
print("\n第五步 ICP 的一步")
r0_, r1_, ang = icp_one_step()
print(f"  源 = 目标转 15°、平移 0.1：初始 RMS {r0_:.4f}；一次“最近邻对应 + Kabsch”后 {r1_:.4f}，还差 {ang:.2f}°")

## 3. 代码里才看得到的

1. **`eigh` 的特征值从小到大排**：所以最小特征值的特征向量是 `v[:, 0]`；符号由 `np.sign(v[2, 0])` 统一成朝上。
2. **ICP 只跑了一步**：完整的“循环到收敛”的点到点 ICP 和 Kabsch 一起留给自己写（里程碑 ME-M33）。
3. **一步后的残余转角从旋转矩阵算**：(tr R − 1)/2 = cos θ（第 19 章的 Log）。

## 4. 自己改一改

1. 把 `icp_one_step` 的初始转角改成 5°，一步后还剩多少？
2. 在 `normal_from_neighbors` 里只用 5 个邻居，夹角误差变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> KD 树查邻居，低维 log n，高维换近似；
> 法向量是局部协方差最小的那个方向；
> 去中心做 SVD，R 等于 V diag(1, 1, det) Uᵀ，别得反射；
> 点到面只罚垂直误差，平面上收敛快；
> ICP 找最近点、求位姿、再找，初值要好。